In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # ✅ 非交互式后端（服务器必备）
import matplotlib.pyplot as plt
import mplfinance as mpf
import os
from tqdm import tqdm
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed
warnings.filterwarnings('ignore')

# ===================== 🔥 AutoDL 核心配置（仅修改这里） =====================
WINDOW_SIZE = 60
# ✅ 保存路径：AutoDL 标准路径 → 直接保存在 autodl-tmp/figures
ROOT_PATH = "/root/autodl-tmp/figures"
# ✅ 原始CSV路径：你的日个股数据2.0.csv 放在 autodl-tmp 根目录
CSV_FILE_PATH = "/root/autodl-tmp/日个股数据2.0.csv"

MA_LIST = [5, 10, 20]
IMAGE_SIZE = (2.24, 2.24)
# AutoDL自动使用最大CPU核心数加速
MAX_WORKERS = None  

# ===================== 初始化 =====================
os.makedirs(ROOT_PATH, exist_ok=True)
plt.rcParams['axes.unicode_minus'] = False

# ===================== 数据加载 =====================
def load_data(file_path):
    df = pd.read_csv(file_path)
    df.rename(columns={
        'Trddt':'Date','Opnprc':'Open','Hiprc':'High','Loprc':'Low',
        'Clsprc':'Close','Dnshrtrd':'Volume'
    }, inplace=True)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(['Stkcd','Date']).reset_index(drop=True)
    return df[['Stkcd','Date','Open','High','Low','Close','Volume']]

# ===================== 单张K线图绘制 =====================
def draw_single_chart(window_data, save_path):
    # 计算均线
    adds = []
    for ma in MA_LIST:
        adds.append(mpf.make_addplot(window_data['Close'].rolling(ma).mean(), color='blue', width=0.7))
    
    # 极简配色
    mc = mpf.make_marketcolors(up='red', down='green', volume={'up':'red','down':'green'})
    style = mpf.make_mpf_style(marketcolors=mc, facecolor='white', figcolor='white')
    
    # 极速绘图
    fig, axes = mpf.plot(
        window_data, type='candle', volume=True, addplot=adds,
        figratio=IMAGE_SIZE, style=style, returnfig=True,
        tight_layout=True, update_width_config={'candle_linewidth':0.5}
    )
    
    # 隐藏所有坐标轴
    for ax in axes:
        ax.axis('off')
    
    # 高清无冗余保存
    plt.savefig(
        save_path, dpi=100, bbox_inches='tight', 
        pad_inches=0, facecolor='white', format='png'
    )
    plt.close(fig)

# ===================== 单只股票处理（断点续跑+多进程） =====================
def process_single_stock_task(stock_data, code):
    stock_folder = os.path.join(ROOT_PATH, str(code))
    os.makedirs(stock_folder, exist_ok=True)
    
    if len(stock_data) < WINDOW_SIZE:
        return
    
    # 跳过已生成的图片（断点续跑）
    for i in range(len(stock_data) - WINDOW_SIZE + 1):
        window = stock_data.iloc[i:i+WINDOW_SIZE].set_index('Date')
        start_date = window.index[0].strftime('%Y%m%d')
        img_path = os.path.join(stock_folder, f"{code}_{start_date}.png")
        
        if os.path.exists(img_path):
            continue
            
        draw_single_chart(window, img_path)
    
    return code

# ===================== 主程序 =====================
if __name__ == "__main__":
    # 1. 加载AutoDL上的CSV数据
    print("📊 正在加载 AutoDL 上的股票数据...")
    df = load_data(CSV_FILE_PATH)
    stock_list = df['Stkcd'].unique()
    
    print(f"✅ 总股票数：{len(stock_list)}")
    print(f"🚀 开启多进程加速 | 保存路径：{ROOT_PATH}\n")
    
    # 2. 准备任务
    tasks = []
    for code in stock_list:
        stock_data = df[df['Stkcd'] == code].reset_index(drop=True)
        tasks.append((stock_data, code))
    
    # 3. 多进程并行绘制
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_single_stock_task, data, code) for data, code in tasks]
        
        # 进度条
        for future in tqdm(as_completed(futures), total=len(futures), desc="绘制进度"):
            pass
    
    print("\n🎉 全部完成！图片已保存至 → /root/autodl-tmp/figures")